# 02 — Simulator & Baseline Policies

This notebook:
1. Loads the feature table from notebook 01
2. Derives remaining proxy features (wetland suitability, biodiversity proxy, etc.)
3. Runs baseline policies through the simulator
4. Compares results

The simulator is the core of the project — it takes a **policy** (one action per cell)
and returns **outcomes** (biodiversity gain, carbon gain, cost, constraint penalties).
Later, NSGA-II will evolve neural networks that produce policies optimizing these outcomes.

In [1]:
import sys
sys.path.insert(0, '../src')

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estonia_landuse.data.constants import DATA_PROCESSED, COUNTY_NAME
from estonia_landuse.simulator.features import derive_features
from estonia_landuse.simulator.simulator import score_policy, summarize_policy
from estonia_landuse.simulator.baselines import (
    policy_no_change, policy_protect_biodiversity,
    policy_restore_wetlands, policy_afforest_carbon, policy_mixed
)
from estonia_landuse.simulator.actions import ACTION_NAMES

## 1. Load Feature Table

Load the GeoPackage created in notebook 01.

In [2]:
gdf = gpd.read_file(DATA_PROCESSED / "base_grid.gpkg")
print(f"Loaded: {len(gdf)} cells, {len(gdf.columns)} columns")
print(f"Columns: {list(gdf.columns)}")

Loaded: 2806 cells, 20 columns
Columns: ['OBJECTID', 'GRD_INSPIR', 'TOTAL_24', 'cell_id', 'land_cover_class', 'land_cover_group', 'urban_pct', 'agriculture_pct', 'grassland_pct', 'forest_pct', 'wetland_pct', 'water_pct', 'other_natural_pct', 'naturalness_score', 'carbon_score', 'protected_overlap_pct', 'distance_to_protected_m', 'road_density_km', 'building_count', 'geometry']


## 2. Derive Proxy Features

From the raw features, we compute four additional columns:

- **wetland_suitability** — how suitable is this cell for wetland restoration?
  Based on existing wetland/water presence and low infrastructure.

- **biodiversity_proxy** — rough estimate of ecological value.
  Combines naturalness, protected area proximity, and low road density.

- **opportunity_cost_proxy** — how economically "expensive" is it to change this cell?
  High population, buildings, agriculture, and road access increase cost.

- **constraint_mask** — bitfield marking which actions are physically/legally impossible.
  E.g., you can't afforest a water cell or restore wetland in a city.

In [1]:
gdf = derive_features(gdf)
print("Derived features added.")
print(f"\nWetland suitability: mean={gdf['wetland_suitability'].mean():.3f}")
print(f"Biodiversity proxy: mean={gdf['biodiversity_proxy'].mean():.3f}")
print(f"Opportunity cost: mean={gdf['opportunity_cost_proxy'].mean():.3f}")
print(f"Cells with any constraint: {(gdf['constraint_mask'] > 0).sum()} / {len(gdf)}")

NameError: name 'derive_features' is not defined

In [ ]:
# Visualize derived features
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

gdf.plot(column="wetland_suitability", legend=True, ax=axes[0,0], cmap="Blues")
axes[0,0].set_title("Wetland Suitability")

gdf.plot(column="biodiversity_proxy", legend=True, ax=axes[0,1], cmap="YlGn")
axes[0,1].set_title("Biodiversity Proxy")

gdf.plot(column="opportunity_cost_proxy", legend=True, ax=axes[1,0], cmap="OrRd")
axes[1,0].set_title("Opportunity Cost Proxy")

gdf.plot(column="constraint_mask", legend=True, ax=axes[1,1], cmap="Set1", categorical=True)
axes[1,1].set_title("Constraint Mask (0=unconstrained)")

for ax in axes.flat:
    ax.set_axis_off()
plt.suptitle("Derived Features", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Run Baseline Policies

We test five simple rule-based policies:

| Policy | Strategy |
|--------|----------|
| No change | Do nothing (lower bound) |
| Protect biodiversity | Protect top 20% cells by biodiversity proxy |
| Restore wetlands | Restore top 15% cells by wetland suitability |
| Afforest for carbon | Afforest where forest is low and cost is low |
| Mixed heuristic | Per-cell best action based on thresholds |

A successful neuroevolution demo should find policies that **dominate** these
baselines on the Pareto front (better on some objectives without being worse on others).

In [ ]:
# Generate all baseline policies
baselines = {
    "No change": policy_no_change(gdf),
    "Protect biodiversity": policy_protect_biodiversity(gdf),
    "Restore wetlands": policy_restore_wetlands(gdf),
    "Afforest for carbon": policy_afforest_carbon(gdf),
    "Mixed heuristic": policy_mixed(gdf),
}

# Score each
results = []
for name, actions in baselines.items():
    summary = summarize_policy(gdf, actions)
    summary["policy"] = name
    # Action distribution
    unique, counts = np.unique(actions, return_counts=True)
    dist = {ACTION_NAMES[u]: c for u, c in zip(unique, counts)}
    summary["action_dist"] = str(dist)
    results.append(summary)

results_df = pd.DataFrame(results).set_index("policy")
print(results_df.drop(columns="action_dist").to_string(float_format="{:.4f}".format))
print("\nAction distributions:")
for _, row in results_df.iterrows():
    print(f"  {row.name}: {row['action_dist']}")

## 4. Compare Baselines

Plot trade-offs between objectives. Each point is one baseline policy.
The ideal policy would be top-right (high gains) with low cost.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biodiversity vs Carbon
for name, row in results_df.iterrows():
    axes[0].scatter(row["carbon_gain"], row["biodiversity_gain"], s=100, zorder=5)
    axes[0].annotate(name, (row["carbon_gain"], row["biodiversity_gain"]),
                     textcoords="offset points", xytext=(5, 5), fontsize=9)
axes[0].set_xlabel("Carbon Gain (mean)")
axes[0].set_ylabel("Biodiversity Gain (mean)")
axes[0].set_title("Biodiversity vs Carbon")
axes[0].grid(True, alpha=0.3)

# Gains vs Cost
for name, row in results_df.iterrows():
    total_gain = row["biodiversity_gain"] + row["carbon_gain"]
    axes[1].scatter(row["cost"], total_gain, s=100, zorder=5)
    axes[1].annotate(name, (row["cost"], total_gain),
                     textcoords="offset points", xytext=(5, 5), fontsize=9)
axes[1].set_xlabel("Cost (mean)")
axes[1].set_ylabel("Total Gain (bio + carbon)")
axes[1].set_title("Total Gain vs Cost")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Map a Policy

Visualize the mixed heuristic policy on the map to check spatial patterns make sense.

In [ ]:
# Map the mixed policy
policy_name = "Mixed heuristic"
actions = baselines[policy_name]

gdf["action"] = [ACTION_NAMES[a] for a in actions]

color_map = {
    "no_change": "lightgray",
    "protect": "green",
    "restore_wetland": "steelblue",
    "afforest": "darkgreen",
}

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
for action_name, color in color_map.items():
    subset = gdf[gdf["action"] == action_name]
    if not subset.empty:
        subset.plot(ax=ax, color=color, label=f"{action_name} ({len(subset)})")

ax.legend(loc="upper right")
ax.set_title(f"{policy_name} — {COUNTY_NAME} County")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 6. Save Enhanced Feature Table

Save the table with derived features for use in the optimizer.

In [ ]:
# Drop the temporary 'action' column before saving
save_gdf = gdf.drop(columns=["action"], errors="ignore")
save_gdf.to_file(DATA_PROCESSED / "features_v1_derived.gpkg", driver="GPKG")
save_gdf.drop(columns="geometry").to_parquet(DATA_PROCESSED / "features_v1_derived.parquet", index=False)
print(f"Saved {len(save_gdf)} cells with {len(save_gdf.columns)} columns to {DATA_PROCESSED}")